# W&B Registry — Promote the Real HiLiftAeroML KPI Dataset

The **W&B Registry** is an organisation-level catalogue of promotable artifacts. It sits on top of project-scoped artifacts: you `log_artifact` inside a project as usual, then `link_artifact` into a Registry **Collection** to make the artifact org-visible, aliasable (`production`, `staging`, ...), and discoverable across projects.

The workshop's Registry is called **`high_lift_surrogates`**. This notebook adds a **`datasets`** Collection to it, containing the real HiLiftAeroML KPI CSV summaries that `experiment_tracking.ipynb` consumes.

**What this notebook does**
1. Load the downloaded HiLiftAeroML CSV summaries.
2. Build the same held-out-geometry split used by the training notebook.
3. Log `force_mom_all.csv`, `geo_values_all.csv`, split manifests, and preview images as a W&B dataset artifact.
4. Promote that artifact into `wandb-registry-high_lift_surrogates/datasets:production`.
5. Read it back from the Registry as a sanity check.

**Prerequisites**
- `wandb login` from this environment.
- The small subset downloaded with `scripts/download_hilift_subset.py --include-images --execute`.
- Access to the `wandb_emea` entity, or edit your local W&B env config.

**Run order in the workshop**
`registry.ipynb` (this notebook) → `experiment_tracking.ipynb` → `sweeps.ipynb` → `reports.ipynb`.


## 1. Setup

Third-party imports and section-local constants.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "physicsx_demo").exists())
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import tempfile

import wandb

from physicsx_demo.utils.env import load_local_env, resolve_wandb_target
from physicsx_demo.utils.hilift_kpi import (
    HILIFT_GEOMETRY_COLUMNS,
    HILIFT_TARGET_COLUMNS,
    kpi_frame_summary,
    prepare_hilift_kpi_bundle,
)

load_local_env()
ENTITY, PROJECT = resolve_wandb_target()

REGISTRY_NAME = "high_lift_surrogates"
DATASET_COLLECTION = "datasets"
DATASET_ARTIFACT_NAME = "hilift_kpi_slice"

SEED = 42
N_GEOMETRIES = 80
DATA_DIR = repo_root.parent / "data" / "hiliftaeroml_subset"
if not (DATA_DIR / "force_mom_all.csv").exists():
    DATA_DIR = repo_root / "data" / "hiliftaeroml_subset"

print(f"wandb   = {wandb.__version__}")
print(f"target  = {ENTITY}/{PROJECT}")
print(f"data    = {DATA_DIR.resolve()}")
print(f"Registry path -> wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}")


## 2. Real HiLiftAeroML KPI Dataset

This workshop path uses the lightweight public CSV summaries from HiLiftAeroML:

- `force_mom_all.csv` contains integrated aerodynamic coefficients.
- `geo_values_all.csv` contains the flap/slat design variables.

The model inputs are eight high-lift design variables plus angle of attack. The targets are `cl`, `cd`, and `cm`. Splits are grouped by `GeoID`, so validation and test rows come from held-out aircraft variants rather than shuffled angles from already-seen geometries.


In [ ]:
# Shared real-data loading helpers live in `src/physicsx_demo/utils/hilift_kpi.py`.
# Keeping this code in the package makes the notebook a W&B orchestration layer.


## 3. Load the Dataset and Prepare Registry Media

The artifact contains the real CSVs needed by downstream notebooks. We also log a few existing HiLiftAeroML rendered images so the Registry entry has inspectable visual context without downloading large VTU fields.


In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Missing {DATA_DIR}. Run: python scripts/download_hilift_subset.py --include-images --execute"
    )

bundle = prepare_hilift_kpi_bundle(DATA_DIR, n_geometries=N_GEOMETRIES, seed=SEED)
summary = kpi_frame_summary(bundle.frame)
train_summary = kpi_frame_summary(bundle.train)
val_summary = kpi_frame_summary(bundle.val)
test_summary = kpi_frame_summary(bundle.test)

print(f"all:   {summary}")
print(f"train: {train_summary}")
print(f"val:   {val_summary}")
print(f"test:  {test_summary}")
print(f"geometry columns: {HILIFT_GEOMETRY_COLUMNS}")
print(f"targets: {HILIFT_TARGET_COLUMNS}")

preview_rows = []
for row in bundle.frame.head(20).itertuples(index=False):
    preview_rows.append([getattr(row, "GeoID"), float(getattr(row, "AoA")), float(getattr(row, "cl")), float(getattr(row, "cd")), float(getattr(row, "cm"))])
preview_table = wandb.Table(data=preview_rows, columns=["GeoID", "AoA", "cl", "cd", "cm"])

split_table = wandb.Table(
    data=[
        ["all", *summary.values()],
        ["train", *train_summary.values()],
        ["val", *val_summary.values()],
        ["test", *test_summary.values()],
    ],
    columns=["split", "rows", "geometries", "aoa_min", "aoa_max"],
)

image_paths = sorted(DATA_DIR.glob("geo_*_AoA_*/img_wss_*.png"))[:12]
image_table = wandb.Table(columns=["image_id", "file", "wall_shear_preview"])
for i, image_path in enumerate(image_paths):
    image_table.add_data(i, image_path.name, wandb.Image(str(image_path), caption=image_path.parent.name))

print(f"prepared {len(image_paths)} rendered WSS preview images")


## 4. Log + Promote to the Registry

The artifact is intentionally small: root summary CSVs, split manifests, and a preview image table. The large boundary/volume archives are not part of the live workshop artifact.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="dataset-registration",
    tags=["notebook", "registry", "dataset", "real-hiliftaeroml"],
    name="hilift-kpi-dataset-register-and-promote",
) as run:
    artifact = wandb.Artifact(
        name=DATASET_ARTIFACT_NAME,
        type="dataset",
        description="Real HiLiftAeroML KPI CSV summaries for CL/CD/CM surrogate training.",
        metadata={
            "source": "nvidia/HiLiftAeroML",
            "n_geometries_used_for_training": N_GEOMETRIES,
            "seed": SEED,
            "summary": summary,
            "train_summary": train_summary,
            "val_summary": val_summary,
            "test_summary": test_summary,
            "geometry_columns": HILIFT_GEOMETRY_COLUMNS,
            "target_columns": HILIFT_TARGET_COLUMNS,
        },
    )
    artifact.add_file(str(DATA_DIR / "force_mom_all.csv"), name="force_mom_all.csv")
    artifact.add_file(str(DATA_DIR / "geo_values_all.csv"), name="geo_values_all.csv")
    if (DATA_DIR / "splits").exists():
        artifact.add_dir(str(DATA_DIR / "splits"), name="splits")
    artifact.add(preview_table, "coefficient_preview")
    artifact.add(split_table, "split_summary")
    artifact.add(image_table, "wall_shear_preview_images")

    run.log({
        "dataset/coefficient_preview": preview_table,
        "dataset/split_summary": split_table,
        "dataset/wall_shear_preview_images": image_table,
    })
    logged = run.log_artifact(artifact)
    logged.wait()

    api = wandb.Api(overrides={"entity": ENTITY, "project": PROJECT})
    try:
        registry = api.create_registry(name=REGISTRY_NAME, visibility="organization")
        print(f"created registry: {registry.name}")
    except Exception as exc:
        print(f"registry create skipped (may already exist): {exc}")

    target_path = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}"
    run.link_artifact(logged, target_path=target_path, aliases=["production", "real-kpi", "v1"])

    print(f"Dataset: {DATASET_ARTIFACT_NAME}")
    print(f"linked -> {target_path}   aliases: production, real-kpi, v1")
    print(f"run url: {run.url}")


## 5. Verify — Read the Dataset Back from the Registry

Sanity-check: a fresh downstream run consumes the promoted entry and rebuilds the held-out geometry split from the downloaded artifact directory.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="registry-verify",
    tags=["notebook", "registry", "verify", "real-hiliftaeroml"],
    name="hilift-kpi-dataset-verify",
) as run:
    art = run.use_artifact(f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production")
    download_dir = Path(art.download())
    verified = prepare_hilift_kpi_bundle(download_dir, n_geometries=N_GEOMETRIES, seed=SEED)

    print(f"pulled artifact dir: {download_dir}")
    print(f"all:   {kpi_frame_summary(verified.frame)}")
    print(f"train: {kpi_frame_summary(verified.train)}")
    print(f"val:   {kpi_frame_summary(verified.val)}")
    print(f"test:  {kpi_frame_summary(verified.test)}")

    run.log({
        "dataset/verified_split_summary": wandb.Table(
            data=[
                ["all", *kpi_frame_summary(verified.frame).values()],
                ["train", *kpi_frame_summary(verified.train).values()],
                ["val", *kpi_frame_summary(verified.val).values()],
                ["test", *kpi_frame_summary(verified.test).values()],
            ],
            columns=["split", "rows", "geometries", "aoa_min", "aoa_max"],
        )
    })


## 6. Wrap Up — What to Look at in the W&B UI

**Runs**
- `hilift-kpi-dataset-register-and-promote` (`job_type = dataset-registration`)
- `hilift-kpi-dataset-verify` (`job_type = registry-verify`)

**Registry**
- `wandb-registry-high_lift_surrogates/datasets` contains `hilift_kpi_slice`, aliased `production`, `real-kpi`, and `v1`.

**Lineage graph**
- `hilift_kpi_slice:v0` is produced by the register run and consumed by the verify run.

**Downstream**
- `experiment_tracking.ipynb` resolves the dataset at `wandb-registry-high_lift_surrogates/datasets:production`, trains `TinyTransolverKpi`, and promotes the best checkpoint to `checkpoints:production`.
